# Multi-Model Customer Support Evaluation

This notebook compares four selected instruction-tuned language models on the same customer support dataset:

- Microsoft Phi-3 Mini, 3.8B
- Google Gemma-2-2B-IT, 2B
- Mistral-7B-Instruct-v0.3, 7B
- Qwen2.5-7B-Instruct, 7B

Each model is loaded, evaluated, then unloaded before the next model is loaded. This avoids keeping all models in GPU memory at once.

In [1]:
!pip -q install -U "transformers>=4.45.0" accelerate sentencepiece pandas bert-score rouge-score nltk psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 149.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


In [2]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
import psutil

from transformers import AutoTokenizer, AutoModelForCausalLM

## Runtime Check

Use a GPU runtime. The 7B models need much more memory than Phi-3 and Gemma.

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. In Colab, go to Runtime > Change runtime type > GPU.")

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
BF16 supported: True


## Optional Hugging Face Login

Gemma requires accepting Google's license on Hugging Face before download. If Gemma fails with an access or gated-repo error, log in here, accept the model conditions on Hugging Face, restart the runtime, and run again.

Clear the output of this cell before submission.

In [4]:
# Only run this if Hugging Face asks you to log in or Gemma gives an access error.
# from huggingface_hub import login
# login()

In [5]:
import os
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets first.")

login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


## Load Dataset

The dataset must include: `ID`, `Category`, `Difficulty`, `Customer Prompt`, and `Reference Answer`.

In [6]:
DATASET_URL = "https://raw.githubusercontent.com/hafishafi/customer-support-Dataset/main/dataset.csv"

dataset = pd.read_csv(DATASET_URL)

print("Rows:", len(dataset))
display(dataset.head())

Rows: 100


,ID,Category,Difficulty,Customer Prompt,Reference Answer
0,1,Account Management,Easy,I forgot my password. How can I reset it?,"Click the ""Forgot Password"" option on the logi..."
1,2,Account Management,Easy,I can't log into my account.,Verify your email and password are correct. If...
2,3,Account Management,Easy,My account is locked. What should I do?,Wait for the lock period to expire or contact ...
3,4,Account Management,Easy,How do I change my email address?,"Go to your account settings, update your email..."
4,5,Account Management,Medium,"I changed my email yesterday, and now I can't ...",Try logging in with your new email address. If...


In [7]:
import pandas as pd

DATASET_URL = "https://raw.githubusercontent.com/hafishafi/customer-support-Dataset/main/dataset.csv"

dataset = pd.read_csv(DATASET_URL)

required_columns = ["ID", "Category", "Difficulty", "Customer Prompt", "Reference Answer"]
missing_columns = [col for col in required_columns if col not in dataset.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Rows:", len(dataset))
display(dataset.head())

Rows: 100


,ID,Category,Difficulty,Customer Prompt,Reference Answer
0,1,Account Management,Easy,I forgot my password. How can I reset it?,"Click the ""Forgot Password"" option on the logi..."
1,2,Account Management,Easy,I can't log into my account.,Verify your email and password are correct. If...
2,3,Account Management,Easy,My account is locked. What should I do?,Wait for the lock period to expire or contact ...
3,4,Account Management,Easy,How do I change my email address?,"Go to your account settings, update your email..."
4,5,Account Management,Medium,"I changed my email yesterday, and now I can't ...",Try logging in with your new email address. If...


## Select Models

For a quick test, set `EVALUATION_LIMIT = 3` or `5`. For final results, keep it as `None` so all 100 prompts are evaluated.

In [8]:
SELECTED_MODELS = [
    {
        "label": "Microsoft Phi-3 Mini",
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "size": "3.8B",
        "developer": "Microsoft",
        "dtype": "float16",
        "attn_implementation": "eager"
    },
    {
        "label": "Google Gemma-2-2B-IT",
        "model_id": "google/gemma-2-2b-it",
        "size": "2B",
        "developer": "Google",
        "dtype": "bfloat16",
        "attn_implementation": "eager"
    },
    {
        "label": "Mistral-7B-Instruct-v0.3",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "size": "7B",
        "developer": "Mistral AI",
        "dtype": "float16",
        "attn_implementation": "eager"
    },
    {
        "label": "Qwen2.5-7B-Instruct",
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "size": "7B",
        "developer": "Alibaba Cloud",
        "dtype": "bfloat16",
        "attn_implementation": "eager"
    }
]

EVALUATION_LIMIT = None  # Change to 5 for a quick test, then back to None for final results.
MAX_NEW_TOKENS = 150

test_dataset = dataset.head(EVALUATION_LIMIT).copy() if EVALUATION_LIMIT else dataset.copy()

print("Models:", len(SELECTED_MODELS))
print("Prompts per model:", len(test_dataset))
print("Total generations:", len(SELECTED_MODELS) * len(test_dataset))

Models: 4
Prompts per model: 100
Total generations: 400


## Helper Functions

In [9]:
def resolve_dtype(dtype_name):
    if dtype_name == "bfloat16" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def load_selected_model(model_info):
    model_id = model_info["model_id"]
    dtype = resolve_dtype(model_info.get("dtype", "float16"))

    print(f"\nLoading {model_info['label']} ({model_id})...")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {
        "torch_dtype": dtype,
        "device_map": "auto"
    }

    attn_implementation = model_info.get("attn_implementation")
    if attn_implementation:
        load_kwargs["attn_implementation"] = attn_implementation

    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    except TypeError:
        load_kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)

    model.eval()
    print("Loaded:", model.config.name_or_path)
    return tokenizer, model


def unload_selected_model(model=None, tokenizer=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def build_messages(customer_prompt):
    instruction = (
        "You are a helpful customer support assistant. "
        "Answer the customer's question clearly, accurately, and concisely."
    )

    return [
        {
            "role": "user",
            "content": f"{instruction}\n\nCustomer question:\n{customer_prompt}"
        }
    ]


def generate_response(prompt, model, tokenizer, max_new_tokens=150):
    messages = build_messages(str(prompt))

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)
    except TypeError:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    input_token_count = inputs["input_ids"].shape[-1]

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    latency = time.perf_counter() - start_time

    generated_ids = outputs[0][input_token_count:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    generated_tokens = int(generated_ids.numel())
    response_length = len(response.split())
    tokens_per_second = generated_tokens / latency if latency > 0 else 0

    gpu_peak_memory_gb = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
        if torch.cuda.is_available()
        else 0
    )

    cpu_process_memory_gb = psutil.Process().memory_info().rss / (1024 ** 3)

    return {
        "Response": response,
        "Latency": latency,
        "Generated Tokens": generated_tokens,
        "Response Length": response_length,
        "Tokens/sec": tokens_per_second,
        "GPU Peak Memory GB": gpu_peak_memory_gb,
        "CPU Process Memory GB": cpu_process_memory_gb
    }

## Run All Models

This is the longest cell. It loads one model, runs all prompts, unloads the model, then moves to the next model.

In [10]:
all_results = []

for model_info in SELECTED_MODELS:
    tokenizer = None
    model = None

    try:
        tokenizer, model = load_selected_model(model_info)

        for _, row in test_dataset.iterrows():
            generated = generate_response(
                row["Customer Prompt"],
                model,
                tokenizer,
                max_new_tokens=MAX_NEW_TOKENS
            )

            all_results.append({
                "Model Label": model_info["label"],
                "Model ID": model_info["model_id"],
                "Model Size": model_info["size"],
                "Developer": model_info["developer"],
                "ID": row["ID"],
                "Category": row["Category"],
                "Difficulty": row["Difficulty"],
                "Prompt": row["Customer Prompt"],
                "Reference Answer": row["Reference Answer"],
                **generated
            })

            print(f"{model_info['label']} - completed prompt {row['ID']}")

    finally:
        unload_selected_model(model, tokenizer)

results_df = pd.DataFrame(all_results)

print("Total result rows:", len(results_df))
display(results_df.head())


Loading Microsoft Phi-3 Mini (microsoft/Phi-3-mini-4k-instruct)...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
Qwen2.5-7B-Instruct - completed prompt 1
Qwen2.5-7B-Instruct - completed prompt 2
Qwen2.5-7B-Instruct - completed prompt 3
Qwen2.5-7B-Instruct - completed prompt 4
Qwen2.5-7B-Instruct - completed prompt 5
Qwen2.5-7B-Instruct - completed prompt 6
Qwen2.5-7B-Instruct - completed prompt 7
Qwen2.5-7B-Instruct - completed prompt 8
Qwen2.5-7B-Instruct - completed prompt 9
Qwen2.5-7B-Instruct - completed prompt 10
Qwen2.5-7B-Instruct - completed prompt 11
Qwen2.5-7B-Instruct - completed prompt 12
Qwen2.5-7B-Instruct - completed prompt 13
Qwen2.5-7B-Instruct - completed prompt 14
Qwen2.5-7B-Instruct - completed prompt 15
Qwen2.5-7B-Instruct - completed prompt 16
Qwen2.5-7B-Instruct - completed prompt 17
Qwen2.5-7B-Instruct - completed prompt 18
Qwen2.5-7B-Instruct - completed prompt 19
Qwen2.5-7B-Instruct - completed prompt 20
Qwen2.5-7B-Instruct - completed prompt 21
Qwen2.5-7B-Instruct - completed prompt 22
Qwen2.5-7B-Instruct - completed prompt 23
Qwen2.5-7B

,Model Label,Model ID,Model Size,Developer,ID,Category,Difficulty,Prompt,Reference Answer,Response,Latency,Generated Tokens,Response Length,Tokens/sec,GPU Peak Memory GB,CPU Process Memory GB
0,Microsoft Phi-3 Mini,microsoft/Phi-3-mini-4k-instruct,3.8B,Microsoft,1,Account Management,Easy,I forgot my password. How can I reset it?,"Click the ""Forgot Password"" option on the logi...","To reset your password, please follow these st...",6.566848,137,101,20.862367,7.233950,2.889015
1,Microsoft Phi-3 Mini,microsoft/Phi-3-mini-4k-instruct,3.8B,Microsoft,2,Account Management,Easy,I can't log into my account.,Verify your email and password are correct. If...,I'm sorry to hear that you're having trouble l...,1.704478,44,34,25.814349,7.157774,2.889126
2,Microsoft Phi-3 Mini,microsoft/Phi-3-mini-4k-instruct,3.8B,Microsoft,3,Account Management,Easy,My account is locked. What should I do?,Wait for the lock period to expire or contact ...,I'm sorry to hear that your account is locked....,5.363248,140,102,26.103586,7.233950,2.889980
3,Microsoft Phi-3 Mini,microsoft/Phi-3-mini-4k-instruct,3.8B,Microsoft,4,Account Management,Easy,How do I change my email address?,"Go to your account settings, update your email...","To change your email address, please follow th...",5.777017,150,106,25.964956,7.233950,2.890125
4,Microsoft Phi-3 Mini,microsoft/Phi-3-mini-4k-instruct,3.8B,Microsoft,5,Account Management,Medium,"I changed my email yesterday, and now I can't ...",Try logging in with your new email address. If...,I'm sorry to hear that you're having trouble s...,3.004772,77,56,25.625907,7.172503,2.899868


## BERTScore for All Model Outputs

In [11]:
from bert_score import score

candidate_responses = results_df["Response"].tolist()
reference_answers = results_df["Reference Answer"].tolist()

bert_device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    P, R, F1 = score(
        candidate_responses,
        reference_answers,
        lang="en",
        verbose=True,
        batch_size=8,
        device=bert_device
    )
except RuntimeError as error:
    if bert_device == "cuda" and "out of memory" in str(error).lower():
        print("BERTScore ran out of GPU memory. Retrying on CPU...")
        torch.cuda.empty_cache()
        gc.collect()
        P, R, F1 = score(
            candidate_responses,
            reference_answers,
            lang="en",
            verbose=True,
            batch_size=4,
            device="cpu"
        )
    else:
        raise

results_df["BERTScore Precision"] = P.cpu().numpy()
results_df["BERTScore Recall"] = R.cpu().numpy()
results_df["BERTScore F1"] = F1.cpu().numpy()

print("Mean BERTScore F1:", results_df["BERTScore F1"].mean())

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/63 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/50 [00:00<?, ?it/s]

done in 2.75 seconds, 145.32 sentences/sec
Mean BERTScore F1: 0.86332834


## ROUGE Scores

In [12]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def calculate_rouge(reference, response):
    scores = rouge.score(str(reference), str(response))
    return pd.Series({
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure
    })

rouge_df = results_df.apply(
    lambda row: calculate_rouge(row["Reference Answer"], row["Response"]),
    axis=1
)

results_df = pd.concat([results_df, rouge_df], axis=1)

print("Mean ROUGE-L:", results_df["ROUGE-L"].mean())

Mean ROUGE-L: 0.17155102059143446


## BLEU Scores

In [13]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1

def calculate_bleu(reference, response):
    return sentence_bleu(
        [str(reference).split()],
        str(response).split(),
        smoothing_function=smooth
    )

results_df["BLEU"] = results_df.apply(
    lambda row: calculate_bleu(row["Reference Answer"], row["Response"]),
    axis=1
)

print("Mean BLEU:", results_df["BLEU"].mean())

Mean BLEU: 0.017695874850572932


## Inference Cost Score

This is a custom score, not money cost. It combines latency and peak GPU memory.

In [14]:
results_df["Inference Cost Score"] = (
    results_df["Latency"] * results_df["GPU Peak Memory GB"]
)

## Clean Summary Table by Model

In [15]:
summary_df = (
    results_df
    .groupby(["Model Label", "Model ID", "Model Size", "Developer"], as_index=False)
    .agg(
        Prompts=("ID", "count"),
        Mean_Latency=("Latency", "mean"),
        Mean_Generated_Tokens=("Generated Tokens", "mean"),
        Mean_Response_Length=("Response Length", "mean"),
        Mean_Tokens_per_Second=("Tokens/sec", "mean"),
        Mean_GPU_Peak_Memory_GB=("GPU Peak Memory GB", "mean"),
        Mean_CPU_Process_Memory_GB=("CPU Process Memory GB", "mean"),
        Mean_BERTScore_F1=("BERTScore F1", "mean"),
        Mean_ROUGE_1=("ROUGE-1", "mean"),
        Mean_ROUGE_2=("ROUGE-2", "mean"),
        Mean_ROUGE_L=("ROUGE-L", "mean"),
        Mean_BLEU=("BLEU", "mean"),
        Mean_Inference_Cost_Score=("Inference Cost Score", "mean")
    )
    .sort_values("Mean_BERTScore_F1", ascending=False)
)

display(
    summary_df.style
    .format({
        "Mean_Latency": "{:.3f}",
        "Mean_Generated_Tokens": "{:.1f}",
        "Mean_Response_Length": "{:.1f}",
        "Mean_Tokens_per_Second": "{:.2f}",
        "Mean_GPU_Peak_Memory_GB": "{:.2f}",
        "Mean_CPU_Process_Memory_GB": "{:.2f}",
        "Mean_BERTScore_F1": "{:.4f}",
        "Mean_ROUGE_1": "{:.4f}",
        "Mean_ROUGE_2": "{:.4f}",
        "Mean_ROUGE_L": "{:.4f}",
        "Mean_BLEU": "{:.4f}",
        "Mean_Inference_Cost_Score": "{:.4f}"
    })
    .hide(axis="index")
    .set_caption("Multi-Model Evaluation Summary")
)

Model Label,Model ID,Model Size,Developer,Prompts,Mean_Latency,Mean_Generated_Tokens,Mean_Response_Length,Mean_Tokens_per_Second,Mean_GPU_Peak_Memory_GB,Mean_CPU_Process_Memory_GB,Mean_BERTScore_F1,Mean_ROUGE_1,Mean_ROUGE_2,Mean_ROUGE_L,Mean_BLEU,Mean_Inference_Cost_Score
Microsoft Phi-3 Mini,microsoft/Phi-3-mini-4k-instruct,3.8B,Microsoft,100,3.450,89.3,64.5,25.96,7.18,2.90,0.8719,0.2643,0.0754,0.1902,0.0230,24.8293
Mistral-7B-Instruct-v0.3,mistralai/Mistral-7B-Instruct-v0.3,7B,Mistral AI,100,4.863,123.2,90.9,25.34,13.53,3.50,0.8672,0.2295,0.0657,0.1616,0.0162,65.8160
Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,7B,Alibaba Cloud,100,3.960,109.2,86.0,27.56,14.21,3.55,0.8620,0.2301,0.0647,0.1644,0.0157,56.2751
Google Gemma-2-2B-IT,google/gemma-2-2b-it,2B,Google,100,4.096,94.2,64.0,23.00,4.89,3.18,0.8522,0.2355,0.0612,0.1700,0.0159,20.0534


## Optional Manual Evaluation Columns

In [16]:
results_df["Accuracy"] = ""
results_df["Relevance"] = ""
results_df["Completeness"] = ""
results_df["Helpfulness"] = ""

## Export Results

In [17]:
combined_results_path = "multi_model_customer_support_results.csv"
summary_path = "multi_model_metric_summary.csv"

results_df.to_csv(combined_results_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Saved:", combined_results_path)
print("Saved:", summary_path)

from google.colab import files
files.download(combined_results_path)
files.download(summary_path)

Saved: multi_model_customer_support_results.csv
Saved: multi_model_metric_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>